# financialkit quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cpoli/financialkit/blob/main/notebooks/quickstart.ipynb)

financialkit shows *how* the models of quantitative finance work: every result is a dataclass you can
inspect, every domain has a plotting helper, and the same conventions hold everywhere (time in years,
rates and volatilities as decimals, an `rng` argument on everything random). Ten minutes, six domains.

*Educational and research software, not investment advice. All data here is synthetic.*

In [ ]:
%pip install -q financialkit  # use "financialkit[fast]" for numba acceleration where numba is available

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import financialkit as fk

print(fk.__version__)

## 1. Options: a binomial tree converging to Black-Scholes

The Cox-Ross-Rubinstein tree converges to the Black-Scholes price at rate $1/n$, oscillating as it goes.

In [ ]:
from financialkit.options.visualizers import plot_tree_convergence

S, K, T, r, sigma = 100.0, 95.0, 1.0, 0.05, 0.2
exact = fk.options.black_scholes_price(S, K, T, r, sigma)
steps = np.arange(5, 301)
prices = np.array([fk.options.CRRTree(n).price(S, K, T, r, sigma).price for n in steps])
print(f"Black-Scholes {exact:.4f}, CRR with 300 steps {prices[-1]:.4f}")
# Black-Scholes 13.3465, CRR with 300 steps 13.3482

ax = plot_tree_convergence(steps, prices, exact)
ax.loglog(steps, 1.0 / steps, "k:", label="1/n")
ax.legend();

## 2. Monte Carlo: a price with an error bar

Monte Carlo returns an estimate and its standard error. The paths come from the shared SDE module, `fk.integrators`.

In [ ]:
mc = fk.monte_carlo.european_option_mc(S, K, T, r, sigma, n_paths=200_000, rng=0)
print(f"{mc.estimate:.4f} +/- {mc.std_error:.4f}; the interval covers Black-Scholes: {mc.covers(exact)}")
# 13.3614 +/- 0.0361; the interval covers Black-Scholes: True

times, paths = fk.integrators.gbm_paths(S, r, sigma, T, n_steps=250, n_paths=30, rng=1)
plt.plot(times, paths.T, lw=0.7)
plt.axhline(K, color="k", ls="--", label="strike")
plt.xlabel("t (years)")
plt.ylabel("stock price")
plt.legend();

## 3. Fixed income: bootstrapping a zero curve

Par rates in, a discount curve out. The curve reprices its inputs exactly.

In [ ]:
from financialkit.fixed_income.visualizers import plot_yield_curve

maturities = np.array([0.5, 1, 2, 3, 5, 7, 10, 20, 30.0])
par_rates = np.array([3.10, 3.20, 3.35, 3.50, 3.70, 3.85, 4.00, 4.25, 4.30]) / 100
curve = fk.fixed_income.bootstrap_par_curve(maturities, par_rates)
print(f"10-year par yield repriced: {100 * curve.par_yield(10.0):.4f}%")
# 10-year par yield repriced: 4.0000%

ax = plot_yield_curve(curve, label="bootstrapped")
ax.plot(maturities, 100 * par_rates, "ko", label="par quotes")
ax.legend();

## 4. Stochastic volatility: where the skew comes from

Under Heston's model, a negative correlation between the stock and its variance turns a flat smile into a skew.

In [ ]:
from financialkit.stochastic_volatility.visualizers import plot_smile

strikes = np.linspace(70, 135, 27)
fig, ax = plt.subplots()
for rho in (-0.7, 0.0, 0.7):
    vols = fk.stochastic_volatility.HestonModel(0.04, 1.5, 0.04, 0.5, rho).implied_vol(100.0, strikes, 1.0, 0.02)
    plot_smile(strikes, vols, forward=100.0 * np.exp(0.02), ax=ax, label=f"rho = {rho}")
ax.legend();

## 5. Portfolios: the efficient frontier

Markowitz's frontier, Tobin's tangency portfolio and the capital market line, for five asset classes.

In [ ]:
from financialkit.portfolio.visualizers import plot_frontier

names = ["bonds", "credit", "US equity", "intl equity", "emerging"]
mu = np.array([0.03, 0.045, 0.07, 0.075, 0.09])
vol = np.array([0.05, 0.08, 0.16, 0.18, 0.25])
corr = np.array(
    [
        [1.0, 0.6, -0.1, -0.1, 0.0],
        [0.6, 1.0, 0.3, 0.3, 0.4],
        [-0.1, 0.3, 1.0, 0.8, 0.7],
        [-0.1, 0.3, 0.8, 1.0, 0.75],
        [0.0, 0.4, 0.7, 0.75, 1.0],
    ]
)
cov = corr * np.outer(vol, vol)
tangency = fk.portfolio.tangency_portfolio(mu, cov, 0.02)
print({n: round(float(w), 3) for n, w in zip(names, tangency.weights, strict=True)})
# {'bonds': 0.484, 'credit': 0.227, 'US equity': 0.168, 'intl equity': 0.12, 'emerging': 0.0}

plot_frontier(fk.portfolio.efficient_frontier(mu, cov, 60, max_return=0.12), labels=names, rf=0.02);

## 6. Risk: volatility clustering and a VaR backtest

Simulate GARCH returns, fit GARCH by maximum likelihood, and backtest a 99% VaR that follows the fitted volatility.

In [ ]:
from financialkit.risk.visualizers import plot_backtest

returns = fk.time_series.GARCH(omega=1e-6, alpha=0.12, beta=0.87).simulate(2000, rng=0).returns
fit = fk.time_series.GARCH.fit(returns)
var = fk.risk.parametric_var(fit.model.mu, np.sqrt(fit.variance), 0.99)
hits = fk.risk.var_exceptions(returns, var)
kupiec = fk.risk.kupiec_test(hits, 0.99)
print(f"{hits.sum()} exceptions, {kupiec.expected:.0f} expected; Kupiec p-value {kupiec.p_value:.2f}")
# 16 exceptions, 20 expected; Kupiec p-value 0.35

plot_backtest(returns, var);

## Where next

* The [example gallery](https://cpoli.github.io/financialkit/) has over 200 runnable examples across 14 domains, each with a *JupyterLite* button.
* Five [tutorials](https://cpoli.github.io/financialkit/tutorials/one_option_five_pricers.html) connect the domains, starting with *One option, five pricers*.
* Each domain's [history page](https://cpoli.github.io/financialkit/history/index.html) walks through the field's breakthroughs in order, from Fibonacci to VPIN, each linked to code that reproduces it.